# 01 — Data Exploration

Walk through the rates / vol panels generated by `make_mock_store` — what
each table looks like at a snapshot, how the time series behave, how curve
spreads move, and which `(expiry, maturity)` pairs each source covers. The
notebook is the local-only entry point: no Bloomberg files required.

## 1. Imports and setup

Add the project root to `sys.path` so `src.*` is importable regardless of
where the Jupyter kernel was launched. Set a seaborn-whitegrid matplotlib
style for legible axes.

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
root = nb_dir if (nb_dir / "src").exists() else nb_dir.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Load the mock store

`make_mock_store` builds the four raw tables (`rate`, `atm_vol`,
`skew_p2`, `skew_n2`) with joint OU dynamics and then computes the two
derived tables. `summary()` is the quick health check: six rows, one per
table, with date range and panel density.

In [ ]:
from src.loaders.mock_loader import make_mock_store
from src.schema import EXPIRY_ORDER, MATURITY_ORDER

store = make_mock_store(n_days=500)
store.summary()

## 3. Rate surface — last-date snapshot

Pivot the rate panel for the most recent date into an `expiry × maturity`
grid. In the mock generator rates are broadcast across expiries (same
value per maturity), so rows are flat by construction — the grid view is
still the right starting picture for when Bloomberg forward rates replace
this later.

In [ ]:
rate_panel = store.as_panel("rate", dropna_threshold=1.0)
last_date = rate_panel.index[-1]
snap = rate_panel.loc[last_date].unstack("maturity")
snap = snap.reindex(
    index=[e for e in EXPIRY_ORDER if e in snap.index],
    columns=[m for m in MATURITY_ORDER if m in snap.columns],
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(snap, annot=True, fmt=".2f", cmap="viridis",
            cbar_kws={"label": "Rate (%)"}, ax=ax)
ax.set_title(f"Rate Surface — {last_date.date()}")
ax.set_xlabel("Maturity")
ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

## 4. ATM vol surface — last-date snapshot

Same view for ATM normal vol (bps). The vol hump built into the OU
long-run mean shows up as a brighter band around the 9M–1Y expiries.

In [ ]:
vol_panel = store.as_panel("atm_vol", dropna_threshold=1.0)
snap_v = vol_panel.loc[last_date].unstack("maturity")
snap_v = snap_v.reindex(
    index=[e for e in EXPIRY_ORDER if e in snap_v.index],
    columns=[m for m in MATURITY_ORDER if m in snap_v.columns],
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(snap_v, annot=True, fmt=".1f", cmap="magma",
            cbar_kws={"label": "ATM vol (bps)"}, ax=ax)
ax.set_title(f"ATM Vol Surface — {last_date.date()}")
ax.set_xlabel("Maturity")
ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

## 5. Time series of representative pairs

Five `(expiry, maturity)` pairs spanning short/long option expiries and
short/long underlying maturities. Long-maturity series sit higher
(upward curve); persistence over weeks is the OU mean-reversion signal.

In [ ]:
target_pairs = [("3M", "2Y"), ("3M", "5Y"), ("3M", "10Y"), ("1Y", "5Y"), ("1Y", "10Y")]

fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
for ax, (exp, mat) in zip(axes, target_pairs):
    s = store.get("rate", expiry=exp, maturity=mat).droplevel(["expiry", "maturity"])
    ax.plot(s.index, s.values, linewidth=1.2, color="C0")
    ax.set_title(f"Rate — expiry {exp}, maturity {mat}")
    ax.set_ylabel("Rate (%)")
    ax.grid(True, alpha=0.4)
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 6. Curve spread time series

`curve_spreads` returns the three classic slope measures in bps. The
zero line is the inversion threshold; persistent crossings are the
regime signals the HMM in `03_regime_detection` will pick up.

In [ ]:
from src.features.derived import curve_spreads

spreads = curve_spreads(store)

fig, ax = plt.subplots(figsize=(12, 5))
spreads[["2s10s", "2s5s", "5s30s"]].plot(ax=ax, linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_ylabel("Spread (bps)")
ax.set_title("Curve spreads")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## 7. Data coverage

A binary `expiry × maturity` grid showing which pairs each raw table
actually has. In the mock data all four raw tables share identical
coverage (the tenor filter is symmetric). On real Bloomberg pulls the
rate source typically covers more pairs than the vol source — this
view is where that gap will become visible.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, name in zip(axes.flat, ["rate", "atm_vol", "skew_p2", "skew_n2"]):
    available = set(store.available_pairs(name))
    grid = pd.DataFrame(
        [[1 if (e, m) in available else 0 for m in MATURITY_ORDER] for e in EXPIRY_ORDER],
        index=EXPIRY_ORDER, columns=MATURITY_ORDER, dtype=int,
    )
    sns.heatmap(grid, ax=ax, cbar=False, cmap="Greys",
                vmin=0, vmax=1, linewidths=0.5, linecolor="white")
    ax.set_title(f"{name} coverage ({int(grid.values.sum())} / {grid.size} pairs)")
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

## 8. Per-pair statistics

Mean / std / min / max / lag-1 autocorrelation per `(expiry, maturity)`
pair, for every table. The colour gradient makes outlier pairs — ones
qualitatively different from their neighbours — easy to spot. Strong
autocorrelation is expected for OU-driven series.

In [ ]:
from IPython.display import display

def per_pair_stats(series: pd.Series) -> pd.DataFrame:
    grp = series.groupby(level=["expiry", "maturity"])
    stats = grp.agg(["mean", "std", "min", "max"])
    stats["autocorr"] = grp.apply(lambda s: s.autocorr(lag=1))
    return stats

for name in ["rate", "atm_vol", "skew_p2", "skew_n2", "skew_spread", "skew_mid"]:
    print(f"\n=== {name} ===")
    stats = per_pair_stats(store.get(name))
    display(stats.style.background_gradient(cmap="coolwarm").format("{:.3f}"))